In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import random
import json
import os
import time
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import DirichletPartitioner,PathologicalPartitioner
from flwr_datasets.preprocessor import Divider

In [ ]:
# Experiment settings. run_experiments.sh overrides them with papermill (this cell is tagged "parameters").
partition = "dirichlet"     # dirichlet: label skew controlled by alpha; pathological: each client has only classes_per_client classes
alpha = 0.5                 # Dirichlet concentration: small means skewed labels per client, large (e.g. 100) is close to IID
classes_per_client = 1      # only used by the pathological partition
num_clients = 10            # pathological split: use at least as many clients as classes, so every class is on some client
client_optimizer = "sgd"    # sgd, sgd_momentum, fedprox, adam, adamw, rmsprop, radam, nadam
server_optimizer = "sgd"    # sgd: FedAvg (weighted average); adam: FedAdam (Adam on the averaged client update)
client_lr = 0.03
server_lr = 0.01            # only used by the adam server optimizer
mu = 0.01                   # FedProx proximal strength, only used by the fedprox client optimizer
local_steps = 50            # mini-batch steps per client per round
rounds = 100
SEED = 11
stage = "manual"            # tune / grid / manual: only used to group results in summary.ipynb
run_name = None             # set by run_experiments.sh; otherwise built from the settings above
results_dir = "../runs/results"   # outside the repo, so results never get in the way of git pull

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

setting = f"alpha{alpha}" if partition=="dirichlet" else f"c{classes_per_client}"
if run_name is None:
    run_name = f"femnist_{stage}_{setting}_{client_optimizer}_{server_optimizer}_clr{client_lr}_slr{server_lr}_mu{mu}_seed{SEED}"
print(run_name)

First we just load and visualize the data from FEMNIST<br>
FEMNIST has only a "train" split: the Divider keeps 80% for the clients and 20% as the server's test set.

In [ ]:
num_classes = 62

if partition=="dirichlet":
    partitioner = DirichletPartitioner(num_partitions=num_clients, partition_by="character",
                                       alpha=alpha, min_partition_size=10,
                                       self_balancing=False,seed=SEED)
else:
    # each client gets exactly classes_per_client classes; with "first-deterministic" client i always has class i
    partitioner = PathologicalPartitioner(num_partitions=num_clients, partition_by="character",
                                          num_classes_per_partition=classes_per_client,
                                          class_assignment_mode="first-deterministic", seed=SEED)

fds = FederatedDataset(dataset="flwrlabs/femnist",
                       preprocessor=Divider(divide_config={"train": 0.8,"test": 0.2,}, divide_split="train",),
                       partitioners={"train": partitioner},
                       seed=SEED)   # rows are shuffled first (FEMNIST rows are grouped by writer)

In [ ]:
train_ds = fds.load_split("train")
test_ds = fds.load_split("test")
print(f"Samples in train - {len(train_ds)}, Samples in test - {len(test_ds)}")
print("The keys of dictionary are-\n",train_ds[0].keys())

fig,axs=plt.subplots(10,10, tight_layout = True)
for i in range(10):
    for j in range(10):
        axs[i,j].imshow(train_ds[i*10+j]["image"], cmap='gray')
        axs[i,j].set_xticks([])
        axs[i,j].set_yticks([])
        axs[i,j].text(0.2, 0.2, train_ds[i*10+j]["character"], horizontalalignment='center', verticalalignment='center', transform=axs[i,j].transAxes)
plt.show()

<h3> Task1</h3>
The training samples are given to each client using a dirichlet prior over <b>class proportions per client.</b><br>
Below: how many samples of each class every client received.

In [ ]:
counts = np.zeros((num_clients, num_classes), dtype=int)   # counts[i][c] = samples of class c on client i
for i in range(num_clients):
    labels = np.array(fds.load_partition(split="train",partition_id=i)["character"])
    counts[i] = np.bincount(labels, minlength=num_classes)
print("Distribution of data samples per client is", counts.sum(axis=1).tolist())

plt.figure(figsize=(12,4))
plt.imshow(counts, aspect="auto", cmap="viridis")
plt.colorbar(label="Number of samples")
plt.xlabel("Class")
plt.ylabel("Client ID")
plt.title(f"Class distribution per client ({setting})")
plt.show()

In [ ]:
class FemnistCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        # Input: (N, 1, 28, 28)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, stride=2)  # halves H and W
        self.dropout2d = nn.Dropout2d(p=0.25)

        # After two pools: 28 -> 14 -> 7, channels = 64
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(128, num_classes)

        for layer in [self.conv1, self.conv2, self.fc1, self.fc2]:
            nn.init.kaiming_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        # x: (N, 1, 28, 28)
        x = self.pool(F.relu(self.conv1(x)))   # (N, 32, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))   # (N, 64, 7, 7)
        x = self.dropout2d(x)

        x = torch.flatten(x, 1)  # (N, 64*7*7)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)          # (N, num_classes)
        return x

model = FemnistCNN(num_classes).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
test_output = model(torch.randn(4, 1, 28, 28).to(device))
assert(test_output.shape==(4,num_classes))

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.ToTensor(),          # -> (1, 28, 28) in [0, 1]
])

class Custom_Dataset(Dataset):
    def __init__(self, dataset, indices, transform=transform):
        # decode every image once and keep the tensors in memory (decoding PIL images at every step is slow)
        subset = dataset.select(indices)
        self.images = torch.stack([transform(img) for img in subset["image"]])   # (N, *(1, 28, 28))
        self.labels = torch.tensor(subset["character"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.images[idx], self.labels[idx])

class client():
    def __init__(self,client_id:int,dataset:Custom_Dataset, model:FemnistCNN, optimizer:str, lr:float, mu:float=0.01):
        self.client_id=client_id
        self.dataset=dataset
        # drop_last keeps every batch full-sized; min(...) keeps at least one batch for very small clients
        self.dataloader=DataLoader(self.dataset, batch_size=min(64, len(self.dataset)), shuffle=True, drop_last=True)
        self.model=model
        self.lr=lr
        self.name=optimizer.lower()
        self.u=mu                 # FedProx proximal strength
        self.fedprox=False        # set to True by _get_optimizer("fedprox"), so it must come before that call
        self.optimizer=self._get_optimizer(self.name)
        self.losses=[]            # cross-entropy per local step (without the FedProx term)

    def _get_optimizer(self, name):
        params=self.model.parameters()
        if name=="sgd":
            return optim.SGD(params, lr=self.lr)
        elif name == "sgd_momentum":
            return optim.SGD(params, lr=self.lr, momentum=0.9, nesterov=False)
        elif name == "adam":
            return optim.Adam(params, lr=self.lr)
        elif name == "adamw":
            return optim.AdamW(params, lr=self.lr)
        elif name == "rmsprop":
            return optim.RMSprop(params, lr=self.lr)
        elif name == "radam":
            return optim.RAdam(params, lr=self.lr)
        elif name=="nadam":
            return optim.NAdam(params, lr=self.lr)
        elif name=="fedprox":
            self.fedprox=True
            return optim.SGD(params, lr=self.lr)
        else:
            raise ValueError(f"Unknown optimizer: {name}")

    def train(self,local_steps, status=False):
        # fresh optimizer every round: clients do not carry momentum / Adam state from one round to the next
        self.optimizer=self._get_optimizer(self.name)
        # FedProx anchor: frozen copy of the global weights received this round
        w_ = {name: param.detach().clone() for name, param in self.model.named_parameters()}
        self.model.train()
        i=0
        while i<local_steps:          # go over the local data as many times as needed for local_steps steps
            for x,y in self.dataloader:
                x=x.to(device)
                y=y.to(device)
                i+=1
                outputs=self.model(x)
                ce=torch.nn.CrossEntropyLoss()(outputs,y)
                loss=ce
                if (self.fedprox):
                    # use live parameters (state_dict() is detached -> zero gradient)
                    for name,param in self.model.named_parameters():
                        loss = loss + (self.u/2)*((param-w_[name])**2).sum()
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                self.losses.append(ce.item())
                if (status and (i%10==0)):
                    print(f"Client {self.client_id} | Step {i}/{local_steps}")
                if i==local_steps:
                    break

    def send_to_server(self):
        state = {name: param.clone() for name, param in self.model.state_dict().items()}
        n = len(self.dataset)
        return (state,n)

    def receive_from_server(self, param_dict):
        state={name: param.clone() for name, param in param_dict.items()}
        self.model.load_state_dict(state)

    def set_learning_rate(self, lr):
        for group in self.optimizer.param_groups:
            group['lr']=lr

class server():
    def __init__(self, clients:[client], dataset:Custom_Dataset, model, optimizer:str, lr:float, local_steps:int, path:str, config:dict=None):
        self.clients=clients
        self.optimizer=optimizer.lower()
        self.model=model
        self.dataset=dataset
        self.dataloader=DataLoader(self.dataset, batch_size=1280, shuffle=False)
        self.lr=lr                    # server learning rate (used by adam)
        self.local_steps=local_steps
        self.path=path                # results are saved as <path>.json and <path>.pt
        self.config=config or {}
        self.losses=[]                # test loss per round
        self.predictions_correct=[]   # test accuracy per round
        self.train_losses=[]          # mean client training loss per round
        self.drift=[]                 # mean L2 distance of the client models from the global model per round
        self.dissimilarity=[]         # spread of the client updates relative to their average per round
        self.round_times=[]           # seconds per round
        self.optimizer_dict=None
        # trainable weights only (a model may also have buffers, e.g. BatchNorm running statistics)
        self.param_names=[name for name,_ in self.model.named_parameters()]
        os.makedirs(os.path.dirname(self.path) or ".", exist_ok=True)

    def global_round(self):
        for client in self.clients:
            client.receive_from_server(self.model.state_dict())
        #put here random indices for client sampling
        for client in self.clients:
            client.train(self.local_steps)
        s=0
        client_weights=[]
        for client in self.clients:
            client_weights.append(client.send_to_server())
            s+=client_weights[-1][1]

        # client drift: how far and how differently the clients moved away from the global model this round
        # dissimilarity = mean ||delta_k - mean delta||^2 / ||mean delta||^2  (0 = all clients agree)
        global_state=self.model.state_dict()
        deltas=[torch.cat([(state[k]-global_state[k]).flatten() for k in self.param_names]) for state,_ in client_weights]
        mean_delta=torch.stack(deltas).mean(0)
        self.drift.append(float(np.mean([d.norm().item() for d in deltas])))
        self.dissimilarity.append(float(np.mean([((d-mean_delta).norm()**2).item() for d in deltas])/((mean_delta.norm()**2).item()+1e-12)))
        self.train_losses.append(float(np.mean([np.mean(c.losses[-self.local_steps:]) for c in self.clients])))

        # weighted mean over clients' total samples
        # integer buffers (e.g. BatchNorm's num_batches_tracked, if a model has one) are copied, not averaged
        new_weight_dict={k:torch.zeros_like(v) for k,v in client_weights[0][0].items()}
        for i in range(len(client_weights)):
            client_dict=client_weights[i][0]
            client_samples=client_weights[i][1]
            for k,v in client_dict.items():
                if v.is_floating_point():
                    new_weight_dict[k]+=v*client_samples/s
                else:
                    new_weight_dict[k]=v.clone()

        if self.optimizer=="sgd":
            self.model.load_state_dict(new_weight_dict)

        elif self.optimizer=="adam":
            if (self.optimizer_dict is None):
                self.optimizer_dict={}
            self.optimizer_dict.setdefault('learning_rate', self.lr)
            self.optimizer_dict.setdefault('beta1', 0.9)
            self.optimizer_dict.setdefault('beta2', 0.999)
            self.optimizer_dict.setdefault('epsilon', 1e-3)   # as in Reddi et al. (Adaptive Federated Optimization)
            self.optimizer_dict.setdefault('m', {k:torch.zeros_like(new_weight_dict[k]) for k in self.param_names})
            self.optimizer_dict.setdefault('v', {k:torch.zeros_like(new_weight_dict[k]) for k in self.param_names})
            self.optimizer_dict.setdefault('t', 0)
            self.optimizer_dict['t']+=1
            m=self.optimizer_dict['m']
            v=self.optimizer_dict['v']
            # Adam only on trainable weights; buffers (if any) keep the plain average
            for k in self.param_names:
                w=global_state[k]
                delta=new_weight_dict[k]-w      # pseudo-gradient: already points downhill
                m[k]=self.optimizer_dict['beta1']*m[k]+(1-self.optimizer_dict['beta1'])*delta
                v[k]=self.optimizer_dict['beta2']*v[k]+(1-self.optimizer_dict['beta2'])*delta*delta
                m_ub=m[k]/(1-self.optimizer_dict['beta1']**self.optimizer_dict['t'])
                v_ub=v[k]/(1-self.optimizer_dict['beta2']**self.optimizer_dict['t'])
                new_weight_dict[k]=w+self.optimizer_dict['learning_rate']*m_ub/(self.optimizer_dict['epsilon']+v_ub.sqrt())
            self.model.load_state_dict(new_weight_dict)
        else:
            raise ValueError(f"Unknown global optimizer: {self.optimizer}")

    def train(self, rounds):
        for i in range(rounds):
            start=time.time()
            self.global_round()
            self.evaluate()
            self.round_times.append(time.time()-start)
            print(f"Round {i+1}/{rounds} | test loss {self.losses[-1]:.4f} | test acc {self.predictions_correct[-1]*100:.2f}% | "
                  f"drift {self.drift[-1]:.3f} | dissimilarity {self.dissimilarity[-1]:.2f} | {self.round_times[-1]:.1f}s")
            self.save_state()
        self.save_state(finished=True)

    @torch.no_grad()
    def evaluate(self):
        # full test set, loss averaged per sample
        self.model.eval()
        loss=0
        correct=0
        total=0
        for x,y in self.dataloader:
            x=x.to(device)
            y=y.to(device)
            outputs=self.model(x)
            loss+=torch.nn.CrossEntropyLoss()(outputs,y).item()*y.shape[0]
            correct+=(outputs.argmax(axis=1)==y).sum().item()
            total+=y.shape[0]
        self.losses.append(loss/total)
        self.predictions_correct.append(correct/total)

    def save_state(self, finished=False):
        torch.save(self.model.state_dict(), f"{self.path}.pt")
        results={"config":self.config, "finished":finished,
                 "test_accuracy":self.predictions_correct, "test_loss":self.losses, "train_loss":self.train_losses,
                 "client_drift":self.drift, "update_dissimilarity":self.dissimilarity, "seconds_per_round":self.round_times}
        with open(f"{self.path}.json", 'w') as file:
            file.write(json.dumps(results))

    def load_state(self):
        self.model.load_state_dict(torch.load(f"{self.path}.pt", weights_only=True))
        with open(f"{self.path}.json", 'r') as file:
            results=json.loads(file.read())
        self.predictions_correct=results["test_accuracy"]
        self.losses=results["test_loss"]
        self.train_losses=results["train_loss"]
        self.drift=results["client_drift"]
        self.dissimilarity=results["update_dissimilarity"]
        self.round_times=results["seconds_per_round"]

In [ ]:
config=dict(dataset="femnist", stage=stage, partition=partition, alpha=alpha, classes_per_client=classes_per_client, setting=setting,
            client_optimizer=client_optimizer, server_optimizer=server_optimizer,
            client_lr=client_lr, server_lr=server_lr, mu=mu, local_steps=local_steps, rounds=rounds, seed=SEED, num_clients=num_clients)

clients=[client(i+1,Custom_Dataset(fds.load_partition(split="train",partition_id=i),list(range(len(fds.load_partition(split="train",partition_id=i))))),FemnistCNN(num_classes).to(device),client_optimizer,client_lr,mu) for i in range(num_clients)]
myServer=server(clients,Custom_Dataset(fds.load_split("test"),list(range(len(fds.load_split("test"))))),FemnistCNN(num_classes).to(device),server_optimizer,server_lr,local_steps,os.path.join(results_dir,run_name),config)

In [ ]:
myServer.train(rounds)

In [ ]:
fig,axs=plt.subplots(1,3, figsize=(15,4))
axs[0].plot(myServer.predictions_correct)
axs[0].set_title("Test accuracy")
axs[1].plot(myServer.train_losses, label="train (clients)")
axs[1].plot(myServer.losses, label="test")
axs[1].set_title("Loss")
axs[1].legend()
axs[2].plot(myServer.drift)
axs[2].set_title("Client drift (mean ||w_k - w||)")
for ax in axs:
    ax.set_xlabel("Round")
plt.tight_layout()
plt.show()